# LAB: Chaos Agent: Tools, Lore, Questionable Judgment

## Objective

Build a **creative complaint handler** for *NormalObjects* with a LangChain agent.
The agent picks its own tools, so you must **prove** the answer comes from tool output.

```
complaint → agent (LLM) → picks tools (any order) → tool outputs → final answer
```

## Steps

| # | Step | Code |
|---|------|---|
| 1 | Setup | PROVIDED |
| 2 | Create creative tools | PROVIDED + YOUR TURN |
| 3 | Create the agent | YOUR TURN |
| 4 | Test with sample complaints | YOUR TURN |
| 5 | Analyze agent behavior | YOUR TURN |
| | `lab_proof.md` | YOUR TURN |

**Submit:** `lab_proof.md` with code, one complaint, tool evidence, final output, one failure.

| Label | Meaning |
|---|---|
| **PROVIDED** | Run and read it |
| **YOUR TURN** | Replace the `# TODO` lines |
| **OPTIONAL** | Not required |

**Reference:** `langchain_integration.ipynb` (tools + `create_agent`)

**Prompt discipline:** use AI to review or debug, not to write your answer.

---
## Step 1: Setup [PROVIDED]

Add `OPENAI_API_KEY=...` to a `.env` file next to this notebook.

In [1]:
# Install required packages (uncomment to run)
!pip install -U langchain langchain-openai python-dotenv

In [2]:
import os
import random
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

load_dotenv()

# Make sure OPENAI_API_KEY is set in your .env file
print("OpenAI API Key loaded:", "OPENAI_API_KEY" in os.environ)

model = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0.7)

OpenAI API Key loaded: True


**Done when:** the key prints `True` and there are no import errors.

---
## Step 2: Create Creative Tools

Every tool needs:
1. the `@tool` decorator
2. a clear docstring (the agent reads it to choose the tool)

### 2a. `consult_demogorgon` [PROVIDED]

In [3]:
@tool
def consult_demogorgon(complaint: str) -> str:
    """Consult the Demogorgon about a complaint involving the Upside Down.
    Returns a chaotic, unreliable opinion."""
    responses = [
        f"The Demogorgon sniffs '{complaint}' and roars. It seems... hungry, not helpful.",
        f"The Demogorgon tilts its petal-face at '{complaint}'. Interpreted as: 'turn off the lights'.",
        f"The Demogorgon ignores '{complaint}' and walks back into the wall. Classic.",
    ]
    return random.choice(responses)

### 2b. `check_hawkins_records` [YOUR TURN]

- Dictionary with 4 keys: `portal`, `monsters`, `psychics`, `electricity`
- Return the record whose key appears in the query (case-insensitive)
- No match → `"No record found ..."`

In [4]:
@tool
def check_hawkins_records(query: str) -> str:
    """Search Hawkins records for a known topic.

    Supported topics are portals, monsters, psychics, and electricity.
    Returns the first matching record, or a no-record message when the
    query does not mention one of those topics.
    """
    records = {
        "portal": "Hawkins records report that portal activity is strongest near the old laboratory and may connect to the Upside Down.",
        "monsters": "Hawkins records warn that monsters can cross from the Upside Down and are unusually active around fresh portal activity.",
        "psychics": "Hawkins records note that psychics may sense or influence activity across the boundary between Hawkins and the Upside Down.",
        "electricity": "Hawkins records link flickering lights and electrical surges to nearby Upside Down activity.",
    }

    query_lower = query.lower()
    for topic, record in records.items():
        if topic in query_lower:
            return record

    return f"No record found for query: {query}"

### 2c. `cast_interdimensional_spell` [YOUR TURN]

- Args: `problem`, `creativity_level="medium"`
- Map `low/medium/high` → `1/2/3`
- 4 spell strings that include the problem
- `random.sample()` N spells, join with `"\n"`

In [5]:
@tool
def cast_interdimensional_spell(problem: str, creativity_level: str = "medium") -> str:
    """Cast creative spells to solve an interdimensional problem.

    The creativity level controls how many possible solutions are returned:
    low returns one, medium returns two, and high returns three.
    """
    creativity_levels = {"low": 1, "medium": 2, "high": 3}
    multiplier = creativity_levels.get(creativity_level.lower(), creativity_levels["medium"])
    spells = [
        f"Summon a benevolent portal to relocate the problem: {problem}.",
        f"Enchant the nearest household object to distract the dimension from: {problem}.",
        f"Trade one embarrassing secret to the Upside Down in exchange for solving: {problem}.",
        f"Reverse the polarity of the Christmas lights and seal away: {problem}.",
    ]
    selected_spells = random.sample(spells, k=min(multiplier, len(spells)))
    return "\n".join(selected_spells)

### 2d. `gather_party_wisdom` [YOUR TURN]

- Dictionary with 4 keys, one per party member (Mike, Dustin, Lucas, Will)
- Return the match (case-insensitive)
- No match → generic "the party huddles..." answer

In [6]:
@tool
def gather_party_wisdom(question: str) -> str:
    """Ask the Hawkins party for advice about a question.

    Returns the response from a named party member when that name appears
    in the question, or a generic response from the whole party.
    """
    party_wisdom = {
        "mike": "Mike says to keep the group together and make a plan before confronting the danger.",
        "dustin": "Dustin says the weird clues probably fit together, so investigate the pattern before panicking.",
        "lucas": "Lucas says to stay practical, bring backup, and never underestimate the monster.",
        "will": "Will says he can sense something wrong, so trust the warning and check the Upside Down.",
    }

    question_lower = question.lower()
    for member, response in party_wisdom.items():
        if member in question_lower:
            return response

    return "The party huddles together and agrees to investigate carefully before making a move."

In [7]:
# Test each tool independently with its expected input schema.
print(consult_demogorgon.invoke({"complaint": "My lights keep flickering"}))
print(check_hawkins_records.invoke({"query": "tell me about the portal"}))
print(cast_interdimensional_spell.invoke({"problem": "Demodogs in my garden", "creativity_level": "high"}))
print(gather_party_wisdom.invoke({"question": "what is the plan?"}))

The Demogorgon tilts its petal-face at 'My lights keep flickering'. Interpreted as: 'turn off the lights'.
Hawkins records report that portal activity is strongest near the old laboratory and may connect to the Upside Down.
Enchant the nearest household object to distract the dimension from: Demodogs in my garden.
Reverse the polarity of the Christmas lights and seal away: Demodogs in my garden.
Trade one embarrassing secret to the Upside Down in exchange for solving: Demodogs in my garden.
The party huddles together and agrees to investigate carefully before making a move.


**Done when:** all 4 tools return text, and a query with no matching key returns your fallback.

---
## Step 3: Create the Agent [YOUR TURN]

System prompt must:
- encourage creative tool use
- force at least one tool call
- say: answer **only** from tool results
- ask the agent to list the tools it used

**Hint:** `create_agent(model=..., tools=..., system_prompt=...)` in `langchain_integration.ipynb`.

In [8]:
# Register every creative tool so the agent can choose and combine them.
tools = [
    consult_demogorgon,
    check_hawkins_records,
    cast_interdimensional_spell,
    gather_party_wisdom,
]

system_prompt = """
You are the NormalObjects Creative Complaint Handler.
Solve complaints creatively by using the available tools in any order,
and combine multiple tools when that helps. You must call at least one
tool before answering. Answer only from the results returned by tools;
do not invent facts, records, or solutions. Be clear when a tool reports
that no record was found. End every response with a line listing the tools
you used, exactly as: Tools used: <tool names>.
"""

agent = create_agent(model=model, tools=tools, system_prompt=system_prompt)

print([t.name for t in tools])

['consult_demogorgon', 'check_hawkins_records', 'cast_interdimensional_spell', 'gather_party_wisdom']


**Done when:** the agent is created and prints the 4 tool names.

---
## Step 4: Test with Sample Complaints [YOUR TURN]

Use `config={"recursion_limit": 10}` to stop loops.

In [9]:
complaints = [
    "My Christmas lights keep flickering and I hear noises from the wall. What is going on?",
    "There is a weird portal in my basement. What's the plan?",
    "A Demodog ate my homework. I need a creative solution, creativity level high!",
    "My neighbor's cat is acting weird. Is it from the Upside Down?",  # no record exists
]

# Run every complaint and keep the complete message history for Step 5.
results = []
for complaint in complaints:
    result = agent.invoke(
        {"messages": [HumanMessage(content=complaint)]},
        config={"recursion_limit": 10},
    )
    results.append(result)
    print(f"\nComplaint: {complaint}")
    print(f"Final answer: {result['messages'][-1].content}")


Complaint: My Christmas lights keep flickering and I hear noises from the wall. What is going on?
Final answer: It seems that your Christmas lights flickering and the noises from the wall could be linked to nearby Upside Down activity, as noted in Hawkins records. Unfortunately, the Demogorgon was not very helpful, only expressing hunger!

Here are some creative solutions to address your issue:

1. **Summon a benevolent portal** to safely relocate the problem away from your home. This could help eliminate any negative energy causing the flickering.

2. **Enchant a household object** (like a broom or a pot) to distract the dimension from your home, allowing for a peaceful atmosphere.

3. **Trade one embarrassing secret** to the Upside Down in exchange for resolving the disturbances. This could be a quirky way to deal with the situation!

Choose one of these options, and may your home find peace from the flickering lights and mysterious noises!

Tools used: check_hawkins_records, consul

**Check:** for the cat complaint, does the agent admit "no record", or invent one?

---
## Step 5: Analyze Agent Behavior [YOUR TURN]

### 5a. Track tool usage

For each result: print the tool order (`a → b → c`). Then count total uses per tool.

**Hint:** AI messages have `.tool_calls`; use `collections.Counter`.

In [10]:
from collections import Counter

tool_counts = Counter()
for result_number, result in enumerate(results, start=1):
    order = []
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []) or []:
            tool_name = call["name"]
            order.append(tool_name)
            tool_counts[tool_name] += 1

    order_text = " → ".join(order) or "(no tools)"
    print(f"Complaint {result_number}: {order_text}")

print("\nTotal tool uses:")
for tool_name, count in tool_counts.most_common():
    print(f"{tool_name}: {count}")

Complaint 1: check_hawkins_records → check_hawkins_records → consult_demogorgon → cast_interdimensional_spell
Complaint 2: check_hawkins_records → consult_demogorgon → cast_interdimensional_spell
Complaint 3: cast_interdimensional_spell
Complaint 4: check_hawkins_records → consult_demogorgon

Total tool uses:
check_hawkins_records: 4
consult_demogorgon: 3
cast_interdimensional_spell: 3


### 5b. Trace one answer (grounding)

For one result, print every tool call, its args, and what it returned.

**Hint:** messages with `msg.type == "tool"` hold the tool output.

In [11]:
# Trace the first complaint from tool selection through final answer.
for message in results[0]["messages"]:
    tool_calls = getattr(message, "tool_calls", []) or []
    for call in tool_calls:
        print(f"TOOL CALL: {call['name']}")
        print(f"ARGS: {call.get('args', {})}")

    if getattr(message, "type", None) == "tool":
        tool_name = getattr(message, "name", "unknown tool")
        print(f"TOOL OUTPUT ({tool_name}): {message.content}")

print("\nFINAL ANSWER:")
print(results[0]["messages"][-1].content)

TOOL CALL: check_hawkins_records
ARGS: {'query': 'electricity'}
TOOL CALL: check_hawkins_records
ARGS: {'query': 'monsters'}
TOOL OUTPUT (check_hawkins_records): Hawkins records link flickering lights and electrical surges to nearby Upside Down activity.
TOOL OUTPUT (check_hawkins_records): Hawkins records warn that monsters can cross from the Upside Down and are unusually active around fresh portal activity.
TOOL CALL: consult_demogorgon
ARGS: {'complaint': 'My Christmas lights keep flickering and I hear noises from the wall.'}
TOOL OUTPUT (consult_demogorgon): The Demogorgon sniffs 'My Christmas lights keep flickering and I hear noises from the wall.' and roars. It seems... hungry, not helpful.
TOOL CALL: cast_interdimensional_spell
ARGS: {'problem': 'flickering Christmas lights and noises from the wall', 'creativity_level': 'high'}
TOOL OUTPUT (cast_interdimensional_spell): Summon a benevolent portal to relocate the problem: flickering Christmas lights and noises from the wall.
Ench

**Check:** mark each claim in the final answer as either from a tool or invented.

### 5c. Compare with a structured approach

Write `structured_handler(complaint)`: always calls records → party → spell (`low`), no LLM.

In [12]:
# Fixed workflow: records → party wisdom → one low-creativity spell.
def structured_handler(complaint: str) -> str:
    records = check_hawkins_records.invoke({"query": complaint})
    party = gather_party_wisdom.invoke({"question": complaint})
    spell = cast_interdimensional_spell.invoke(
        {"problem": complaint, "creativity_level": "low"}
    )

    return (
        f"Hawkins records: {records}\n"
        f"Party wisdom: {party}\n"
        f"Suggested spell: {spell}"
    )

print(structured_handler(complaints[1]))

Hawkins records: Hawkins records report that portal activity is strongest near the old laboratory and may connect to the Upside Down.
Party wisdom: The party huddles together and agrees to investigate carefully before making a move.
Suggested spell: Reverse the polarity of the Christmas lights and seal away: There is a weird portal in my basement. What's the plan?.


| | Agent | Structured |
|---|---|---|
| Same tools every run? | No | Yes: records → party → spell |
| Tool order predictable? | No | Yes |
| Can invent facts? | Possible if the model ignores grounding instructions | No LLM, so no generated facts |
| Easy to trace? | Moderate: inspect the message history | Yes: fixed calls and outputs |
| Best for which client case? | Exploratory, creative complaints | Predictable, auditable workflows |

---
## Extensions [OPTIONAL]

- More tools (`consult_eleven`, `check_government_files`)
- LLM call inside a tool
- Memory with `InMemorySaver` + `thread_id`
- Streamlit / Gradio UI
- Measure response time and tool calls per complaint

In [ ]:
# YOUR TURN (optional)

# TODO

---
## Write `lab_proof.md` [YOUR TURN]

```markdown
# Lab Proof: NormalObjects Creative Complaint Handler

## 1. What I built
## 2. Query
## 3. Tool evidence (calls + args + outputs)
## 4. Final output
## 5. Trace (which claim comes from which tool output?)
## 6. Failure or limitation (where can it give a plausible but unsupported answer?)
## 7. Agent vs. structured handler
```

**Before you submit:**
- [ ] No API keys or `.env`
- [ ] Only lab files
- [ ] `lab_proof.md` is the entry point

---
### Troubleshooting

| Problem | Fix |
|---|---|
| Agent doesn't use tools | Clearer docstrings; prompt must require tool use |
| Tools in "wrong" order | Expected for agents. Need fixed order? Use LangGraph |
| Agent loops | Lower `recursion_limit`, improve docstrings |
| Generic answers | More specific tool outputs, lower temperature |
| `ImportError` on `create_agent` | `pip install -U langchain` (needs v1+) |